# Output BAB 4.4.3 — Stabilitas Pelatihan Model

Notebook ini khusus membuat artefak untuk subbab **4.4.3 Stabilitas Pelatihan Model** sesuai `TODO_BAB_4_Lengkap.md` dan `TODO_BAB_4_Checklist_Output.md`. Output mencakup kurva loss/IoU/Dice/LR, tabel titik kurva, ringkasan stabilitas, kebijakan training, narasi interpretasi, dan checklist cakupan TODO.


## 0. Setup

Konfigurasi path, folder output, plotting style, dan helper.


In [ ]:
from __future__ import annotations

import json
import math
import re
from collections import Counter, defaultdict
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

try:
    import rasterio
except Exception as exc:
    rasterio = None
    RASTERIO_IMPORT_ERROR = f"{type(exc).__name__}: {exc}"
else:
    RASTERIO_IMPORT_ERROR = ""

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 300,
    "axes.titlesize": 11,
    "axes.labelsize": 10,
    "font.size": 10,
})

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

DATASET = ROOT / "dataset"
RUNS = ROOT / "runs"
OUT = ROOT / "outputs" / "bab4"
TABLE_DIR = OUT / "tables"
FIG_DIR = OUT / "figures"
NARRATIVE_DIR = OUT / "narratives"
TABLE_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)
NARRATIVE_DIR.mkdir(parents=True, exist_ok=True)

REGIONS = [
    "Aceh_Besar", "Aceh_Tamiang", "Aceh_Timur", "Aceh_Utara", "Agam",
    "Banda_Aceh", "Bireuen", "Langsa", "Pasaman_Barat", "Pidie", "Pidie_Jaya",
]
TEST_REGION = "Aceh_Utara"
CHANNEL_NAMES = ["vv_norm", "vh_norm", "hue", "saturation", "value", "slope_norm", "hand_norm"]

print(f"ROOT: {ROOT}")
print(f"Tables: {TABLE_DIR}")
print(f"Figures: {FIG_DIR}")
print("rasterio:", "available" if rasterio else f"missing ({RASTERIO_IMPORT_ERROR})")


In [ ]:
def display_df(df: pd.DataFrame, max_rows: int = 20):
    if df.empty:
        display(df)
    else:
        display(df.head(max_rows))


def save_table(df: pd.DataFrame, name: str) -> Path:
    path = TABLE_DIR / name
    df.to_csv(path, index=False)
    print(f"saved table: {path.relative_to(ROOT)} ({len(df)} rows)")
    display_df(df)
    return path


def save_fig(fig, name: str) -> Path:
    path = FIG_DIR / name
    try:
        fig.tight_layout(rect=[0, 0, 1, 0.96])
    except Exception:
        fig.tight_layout()
    fig.savefig(path, bbox_inches="tight")
    print(f"saved figure: {path.relative_to(ROOT)}")
    plt.show()
    return path


def read_csv_or_status(path: Path, label: str) -> pd.DataFrame:
    if not path.exists():
        return pd.DataFrame([{"source": str(path.relative_to(ROOT)), "status": "missing", "note": label}])
    try:
        return pd.read_csv(path)
    except Exception as exc:
        return pd.DataFrame([{"source": str(path.relative_to(ROOT)), "status": "read_error", "note": f"{type(exc).__name__}: {exc}"}])


def safe_div(num: float, den: float) -> float:
    return float(num / den) if den else 0.0


def pct(num: float, den: float) -> float:
    return round(100.0 * safe_div(num, den), 4)


def finite_stats(arr: np.ndarray, sample_step: int = 1) -> dict:
    a = np.asarray(arr)
    if sample_step > 1 and a.ndim >= 2:
        a = a[..., ::sample_step, ::sample_step]
    a = a.astype("float64", copy=False)
    finite = np.isfinite(a)
    vals = a[finite]
    if vals.size == 0:
        return {"count": 0, "valid_pct": 0.0, "min": np.nan, "max": np.nan, "mean": np.nan, "std": np.nan, "p2": np.nan, "p98": np.nan}
    return {
        "count": int(vals.size),
        "valid_pct": round(100.0 * vals.size / a.size, 4),
        "min": float(np.min(vals)),
        "max": float(np.max(vals)),
        "mean": float(np.mean(vals)),
        "std": float(np.std(vals)),
        "p2": float(np.percentile(vals, 2)),
        "p98": float(np.percentile(vals, 98)),
    }


def normalize_image(arr: np.ndarray, p_low: float = 2, p_high: float = 98) -> np.ndarray:
    a = np.asarray(arr, dtype="float32")
    finite = np.isfinite(a)
    if not finite.any():
        return np.zeros_like(a, dtype="float32")
    lo, hi = np.percentile(a[finite], [p_low, p_high])
    if not np.isfinite(lo) or not np.isfinite(hi) or hi <= lo:
        lo, hi = float(np.nanmin(a[finite])), float(np.nanmax(a[finite]))
    if hi <= lo:
        return np.zeros_like(a, dtype="float32")
    return np.clip((a - lo) / (hi - lo), 0, 1)


def raster_stats(path: Path, band: int = 1, sample_step: int = 10) -> dict:
    if rasterio is None:
        return {"status": "skipped", "note": f"rasterio unavailable: {RASTERIO_IMPORT_ERROR}"}
    if not path.exists():
        return {"status": "missing", "note": str(path.relative_to(ROOT))}
    try:
        with rasterio.open(path) as src:
            arr = src.read(band, masked=True)
            vals = arr.compressed() if np.ma.isMaskedArray(arr) else np.asarray(arr).ravel()
            if sample_step > 1:
                arr2 = np.asarray(arr.filled(np.nan) if np.ma.isMaskedArray(arr) else arr)
                vals = arr2[::sample_step, ::sample_step].ravel()
            stats = finite_stats(vals)
            stats.update({
                "status": "ok",
                "height": src.height,
                "width": src.width,
                "crs": str(src.crs),
                "transform": str(src.transform),
            })
            return stats
    except Exception as exc:
        return {"status": "read_error", "note": f"{type(exc).__name__}: {exc}"}


def load_npz(path: Path):
    return np.load(path, allow_pickle=False)


def first_npz(path: Path) -> Path | None:
    files = sorted(path.glob("*.npz"))
    return files[0] if files else None


def parse_tile_rc(path: Path) -> tuple[int | None, int | None]:
    m = re.search(r"_r(\d+)_c(\d+)", path.stem)
    if not m:
        return None, None
    return int(m.group(1)), int(m.group(2))


def choose_tile(region: str = TEST_REGION) -> Path | None:
    tile_dir = DATASET / "tiles" / "7ch" / "by_region" / region
    candidates = []
    for path in sorted(tile_dir.glob("*.npz")):
        try:
            z = load_npz(path)
            y = np.asarray(z["y"])[0]
            valid = np.asarray(z.get("valid_mask", np.ones_like(y)))[0]
            positives = int(((y > 0) & (valid > 0)).sum())
            valid_count = int((valid > 0).sum())
            candidates.append((positives, valid_count, path))
        except Exception:
            continue
    if not candidates:
        return None
    positives = [c for c in candidates if c[0] > 0]
    if positives:
        return sorted(positives, key=lambda x: x[0], reverse=True)[0][2]
    return sorted(candidates, key=lambda x: x[1], reverse=True)[0][2]


def find_matching_prediction(model: str, tile_path: Path) -> Path | None:
    pred_dir = RUNS / "final" / model / "eval_test" / "predictions" / TEST_REGION
    candidate = pred_dir / tile_path.name
    if candidate.exists():
        return candidate
    r, c = parse_tile_rc(tile_path)
    if r is None:
        return first_npz(pred_dir)
    matches = sorted(pred_dir.glob(f"*r{r:06d}_c{c:06d}.npz"))
    return matches[0] if matches else first_npz(pred_dir)


## 4.4.3 Load Training Curves

Membaca kurva dari konfigurasi hyperparameter terbaik masing-masing model dan metrik final training sebagai audit.


In [ ]:
def pretty_model(model: str) -> str:
    return {"unet": "U-Net", "procanet": "ProCANet"}.get(str(model).lower(), str(model))


def model_dir_name(model: str) -> str:
    return {"U-Net": "unet", "ProCANet": "procanet"}.get(model, model.lower())


def load_best_configs() -> pd.DataFrame:
    best_path = TABLE_DIR / "4_4_2_best_hyperparameters_by_model.csv"
    if best_path.exists():
        return pd.read_csv(best_path)
    rows = []
    for model, variant in [("U-Net", "grid_lr_5e-5_wd_1e-4"), ("ProCANet", "grid_lr_1e-4_wd_1e-4")]:
        rows.append({"model": model, "variant": variant})
    return pd.DataFrame(rows)


def read_metrics(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path)
    if "epoch" not in df.columns:
        df["epoch"] = np.arange(1, len(df) + 1)
    return df

best_configs = load_best_configs()
selected_rows = []
for _, cfg in best_configs.iterrows():
    model = cfg["model"]
    model_dir = model_dir_name(model)
    variant = cfg["variant"]
    for metrics_path in sorted((RUNS / model_dir).glob(f"fold_*/{variant}/metrics.csv")):
        df = read_metrics(metrics_path)
        rel = metrics_path.relative_to(RUNS)
        fold = rel.parts[1]
        for _, row in df.iterrows():
            rec = row.to_dict()
            rec.update({
                "model": model,
                "run_type": "best_grid_cv",
                "variant": variant,
                "fold": fold,
                "run": "/".join(rel.parts[:-1]),
                "metrics_path": str(metrics_path.relative_to(ROOT)),
            })
            selected_rows.append(rec)

# Final runs have train metrics only; keep them in the same audit table but exclude from train-vs-val plot when validation columns are absent.
for metrics_path in sorted((RUNS / "final").glob("*/metrics.csv")):
    model = pretty_model(metrics_path.parent.name)
    df = read_metrics(metrics_path)
    for _, row in df.iterrows():
        rec = row.to_dict()
        rec.update({
            "model": model,
            "run_type": "final_train_only",
            "variant": "final",
            "fold": "final",
            "run": "/".join(metrics_path.relative_to(RUNS).parts[:-1]),
            "metrics_path": str(metrics_path.relative_to(ROOT)),
        })
        selected_rows.append(rec)

curves = pd.DataFrame(selected_rows)
if curves.empty:
    curves = pd.DataFrame([{"status": "missing", "note": "No metrics found for best hyperparameter curves"}])
save_table(curves, "4_4_3_training_curve_points.csv")


## 4.4.3 Ringkasan Stabilitas

Menghitung best epoch, best validation IoU/Dice/Loss, gap train-validation, penurunan LR, dan early stopping.


In [ ]:
stability_rows = []
if "status" not in curves.columns and not curves.empty:
    cv_curves = curves[curves["run_type"].eq("best_grid_cv")].copy()
    for (model, variant, fold), g in cv_curves.groupby(["model", "variant", "fold"]):
        g = g.sort_values("epoch")
        best_idx = g["val_iou"].idxmax() if "val_iou" in g.columns and g["val_iou"].notna().any() else g.index[-1]
        best = g.loc[best_idx]
        final = g.iloc[-1]
        lr_unique = int(g["lr"].nunique()) if "lr" in g.columns else 0
        lr_reduced = bool(lr_unique > 1)
        stability_rows.append({
            "model": model,
            "variant": variant,
            "fold": fold,
            "epoch_count": int(g["epoch"].max()),
            "best_epoch": int(best.get("epoch", np.nan)),
            "best_val_iou": float(best.get("val_iou", np.nan)),
            "best_val_dice": float(best.get("val_dice", np.nan)),
            "best_val_loss": float(best.get("val_loss", np.nan)),
            "train_iou_at_best": float(best.get("train_iou", np.nan)),
            "train_val_iou_gap_at_best": float(best.get("train_iou", np.nan) - best.get("val_iou", np.nan)),
            "final_train_loss": float(final.get("train_loss", np.nan)),
            "final_val_loss": float(final.get("val_loss", np.nan)),
            "final_train_iou": float(final.get("train_iou", np.nan)),
            "final_val_iou": float(final.get("val_iou", np.nan)),
            "final_lr": float(final.get("lr", np.nan)),
            "lr_reduced": lr_reduced,
            "lr_unique_count": lr_unique,
            "stopped_early": bool(final.get("stopped_early", 0)),
            "bad_epochs_final": int(final.get("bad_epochs", 0)) if not pd.isna(final.get("bad_epochs", np.nan)) else 0,
        })
training_stability = pd.DataFrame(stability_rows)
save_table(training_stability, "4_4_3_training_stability_summary.csv")

if not training_stability.empty:
    model_stability = (
        training_stability.groupby(["model", "variant"], as_index=False)
        .agg(
            fold_count=("fold", "nunique"),
            mean_best_epoch=("best_epoch", "mean"),
            mean_best_val_iou=("best_val_iou", "mean"),
            std_best_val_iou=("best_val_iou", "std"),
            mean_best_val_dice=("best_val_dice", "mean"),
            mean_train_val_iou_gap_at_best=("train_val_iou_gap_at_best", "mean"),
            lr_reduced_folds=("lr_reduced", "sum"),
            stopped_early_folds=("stopped_early", "sum"),
            mean_final_train_loss=("final_train_loss", "mean"),
            mean_final_val_loss=("final_val_loss", "mean"),
        )
    )
else:
    model_stability = pd.DataFrame()
save_table(model_stability, "4_4_3_model_stability_summary.csv")

final_train = curves[curves.get("run_type", pd.Series(dtype=str)).eq("final_train_only")].copy() if "run_type" in curves else pd.DataFrame()
final_rows = []
for model, g in final_train.groupby("model") if not final_train.empty else []:
    g = g.sort_values("epoch")
    first = g.iloc[0]
    last = g.iloc[-1]
    final_rows.append({
        "model": model,
        "epoch_count": int(last.get("epoch", len(g))),
        "initial_train_loss": float(first.get("train_loss", np.nan)),
        "final_train_loss": float(last.get("train_loss", np.nan)),
        "initial_train_iou": float(first.get("train_iou", np.nan)),
        "final_train_iou": float(last.get("train_iou", np.nan)),
        "initial_train_dice": float(first.get("train_dice", np.nan)),
        "final_train_dice": float(last.get("train_dice", np.nan)),
        "note": "final run contains train metrics only; validation stability is read from best-grid CV curves",
    })
final_training_summary = pd.DataFrame(final_rows)
save_table(final_training_summary, "4_4_3_final_training_summary.csv")


## 4.4.3 Grafik Loss, IoU, Dice, dan Learning Rate

Membuat grafik multi-panel training vs validation untuk konfigurasi terbaik.


In [ ]:
def aggregate_metric(df: pd.DataFrame, metric: str) -> pd.DataFrame:
    cols = ["model", "epoch"]

    if metric not in df.columns:
        return pd.DataFrame(columns=cols + ["mean", "std"])

    return df.groupby(cols, as_index=False).agg(
        mean=(metric, "mean"),
        std=(metric, "std")
    )


def add_panel_label(ax, label: str, y: float = -0.22, fontsize: int = 20):
    ax.text(
        0.5, y, label,
        transform=ax.transAxes,
        ha="center",
        va="top",
        fontsize=fontsize,
        clip_on=False
    )


if "status" not in curves.columns and not curves.empty:
    cv = curves[curves["run_type"].eq("best_grid_cv")].copy()

    fig, axes = plt.subplots(2, 2, figsize=(12.5, 8.2))

    metric_specs = [
        ("Loss", "train_loss", "val_loss", axes[0, 0]),
        ("IoU", "train_iou", "val_iou", axes[0, 1]),
        ("Dice/F1", "train_dice", "val_dice", axes[1, 0]),
    ]

    colors = {
        "U-Net": "#C87954",
        "ProCANet": "#5274A5"
    }

    subplot_labels = ["(a)", "(b)", "(c)", "(d)"]

    for idx, (title, train_col, val_col, ax) in enumerate(metric_specs):
        for model, model_df in cv.groupby("model"):
            for col, style, phase in [
                (train_col, "--", "train"),
                (val_col, "-", "validation")
            ]:
                agg = aggregate_metric(model_df, col)

                if agg.empty:
                    continue

                ax.plot(
                    agg["epoch"],
                    agg["mean"],
                    linestyle=style,
                    color=colors.get(model),
                    label=f"{model} {phase}"
                )

        add_panel_label(
            ax,
            subplot_labels[idx],
            y=-0.22,
            fontsize=20
        )

        ax.set_xlabel("Epoch", labelpad=8)
        ax.set_ylabel(title)
        ax.grid(True, alpha=0.25)

    # Learning rate curve.
    ax = axes[1, 1]

    for model, model_df in cv.groupby("model"):
        agg = aggregate_metric(model_df, "lr")

        if not agg.empty:
            ax.plot(
                agg["epoch"],
                agg["mean"],
                color=colors.get(model),
                label=model
            )

    add_panel_label(
        ax,
        subplot_labels[3],
        y=-0.22,
        fontsize=20
    )

    ax.set_xlabel("Epoch", labelpad=8)
    ax.set_ylabel("Learning rate")
    ax.grid(True, alpha=0.25)

    handles, labels = axes[0, 0].get_legend_handles_labels()

    fig.legend(
        handles,
        labels,
        loc="upper center",
        ncol=4,
        bbox_to_anchor=(0.5, 0.995)
    )

    # Tidak memakai fig.suptitle.
    # Tidak memakai tight_layout setelah subplots_adjust,
    # supaya posisi label panel tidak diubah otomatis.
    fig.subplots_adjust(
        left=0.08,
        right=0.98,
        top=0.88,
        bottom=0.14,
        wspace=0.22,
        hspace=0.65
    )

    save_fig(fig, "4_4_3_training_curves.png")


    # Per-model final training-only curves, clearly separated from validation stability.
    final_train = curves[curves["run_type"].eq("final_train_only")].copy()

    if not final_train.empty:
        fig, axes = plt.subplots(1, 3, figsize=(13, 4.2))

        final_metric_specs = [
            ("train_loss", axes[0], "Final Train Loss"),
            ("train_iou", axes[1], "Final Train IoU"),
            ("train_dice", axes[2], "Final Train Dice"),
        ]

        final_subplot_labels = ["(a)", "(b)", "(c)"]

        for idx, (metric, ax, title) in enumerate(final_metric_specs):
            for model, model_df in final_train.groupby("model"):
                ax.plot(
                    model_df["epoch"],
                    model_df[metric],
                    color=colors.get(model),
                    label=model
                )

            add_panel_label(
                ax,
                final_subplot_labels[idx],
                y=-0.30,
                fontsize=20
            )

            ax.set_xlabel("Epoch", labelpad=8)
            ax.set_ylabel(title.replace("Final Train ", ""))
            ax.grid(True, alpha=0.25)

        axes[0].legend(loc="best")

        # Tidak memakai fig.suptitle.
        fig.subplots_adjust(
            left=0.07,
            right=0.98,
            top=0.92,
            bottom=0.28,
            wspace=0.28
        )

        save_fig(fig, "4_4_3_final_training_curves.png")

## 4.4.3 Kebijakan Training

Meringkas optimizer, loss, valid mask, scheduler, early stopping, dan best checkpoint dari implementasi training.


In [ ]:
policy_rows = []
script_path = ROOT / "scripts" / "train_segmentation.py"
loss_path = ROOT / "training" / "losses.py"
train_text = script_path.read_text() if script_path.exists() else ""
loss_text = loss_path.read_text() if loss_path.exists() else ""
policy_rows.extend([
    {"component": "optimizer", "implementation": "AdamW", "evidence": "torch.optim.AdamW in scripts/train_segmentation.py", "status": "ok" if "AdamW" in train_text else "missing"},
    {"component": "loss", "implementation": "masked BCEWithLogits + masked Dice Loss", "evidence": "masked_bce_dice_loss = masked_bce_with_logits + masked_dice_loss", "status": "ok" if "masked_bce_dice_loss" in loss_text and "binary_cross_entropy_with_logits" in loss_text else "missing"},
    {"component": "valid_mask", "implementation": "loss and metrics ignore invalid pixels", "evidence": "valid_mask.bool(); logits[mask]; target[mask]", "status": "ok" if "valid_mask.bool" in loss_text and "logits[mask]" in loss_text else "missing"},
    {"component": "scheduler", "implementation": "ReduceLROnPlateau(mode='max') on validation IoU", "evidence": "scheduler.step(val_metrics['iou']); ReduceLROnPlateau(mode='max')", "status": "ok" if "ReduceLROnPlateau" in train_text and "scheduler.step(val_metrics" in train_text else "missing"},
    {"component": "early_stopping", "implementation": "EarlyStopping uses validation IoU", "evidence": "early_stopping.step(val_metrics['iou'])", "status": "ok" if "early_stopping.step(val_metrics" in train_text else "missing"},
    {"component": "best_checkpoint", "implementation": "best.pt saved when validation IoU improves", "evidence": "save_checkpoint_if_best; best_val_iou", "status": "ok" if "best_val_iou" in train_text and "save_checkpoint_if_best" in train_text else "missing"},
    {"component": "selection_metric", "implementation": "validation IoU", "evidence": "best_val_iou and val_metrics['iou']", "status": "ok" if "val_metrics[\"iou\"]" in train_text or "val_metrics['iou']" in train_text else "missing"},
])
training_policy = pd.DataFrame(policy_rows)
save_table(training_policy, "4_4_3_training_policy_summary.csv")


## 4.4.3 Narasi Interpretasi

Menyimpan paragraf siap pakai untuk menjelaskan checkpoint, masked loss, scheduler, early stopping, dan stabilitas/overfitting.


In [ ]:
def fmt_model_row(model: str) -> str:
    if "model_stability" not in globals() or model_stability.empty:
        return f"{model}: data tidak tersedia"
    row = model_stability[model_stability["model"].eq(model)]
    if row.empty:
        return f"{model}: data tidak tersedia"
    r = row.iloc[0]
    return (
        f"{model}: mean best Val IoU {r['mean_best_val_iou']:.4f}, "
        f"mean best Val Dice {r['mean_best_val_dice']:.4f}, "
        f"rata-rata gap train-val IoU {r['mean_train_val_iou_gap_at_best']:.4f}, "
        f"LR turun pada {int(r['lr_reduced_folds'])}/{int(r['fold_count'])} fold, "
        f"early stopping pada {int(r['stopped_early_folds'])}/{int(r['fold_count'])} fold"
    )

interpretation_text = f"""# Interpretasi Stabilitas Pelatihan 4.4.3

Kurva stabilitas 4.4.3 dibuat dari konfigurasi hyperparameter terbaik masing-masing model pada 5-fold spatial cross-validation. Dengan demikian, kurva loss, IoU, Dice, dan learning rate merepresentasikan perilaku training pada konfigurasi yang dipilih untuk eksperimen utama, bukan seluruh 60 run grid search.

Checkpoint terbaik disimpan berdasarkan validation IoU. Pemilihan ini lebih relevan daripada validation loss karena tujuan akhir segmentasi adalah meningkatkan tumpang tindih spasial antara prediksi dan label. Kebijakan training juga memakai early stopping berbasis validation IoU, sehingga pelatihan dihentikan ketika validation IoU tidak lagi membaik sesuai patience/min_delta.

Loss yang digunakan adalah masked BCEWithLogits + masked Dice Loss. Keduanya memakai `valid_mask`, sehingga piksel di luar area valid tidak memengaruhi optimasi maupun evaluasi metrik. Ini penting karena label UNOSAT memiliki area analisis tertentu dan piksel di luar area tersebut tidak boleh dihitung sebagai sinyal training.

Ringkasan stabilitas menunjukkan {fmt_model_row('U-Net')}. Untuk ProCANet, {fmt_model_row('ProCANet')}. Gap train-val IoU yang bernilai positif besar dapat mengindikasikan kecenderungan overfitting, sedangkan gap kecil atau negatif menunjukkan validasi tidak lebih buruk daripada training pada epoch terbaik. Penurunan learning rate menunjukkan scheduler ReduceLROnPlateau aktif ketika validation IoU mengalami plateau.

Kurva final training disimpan terpisah karena run final hanya memiliki metrik training tanpa validation metrics. Oleh karena itu, interpretasi stabilitas generalisasi tetap mengacu pada kurva cross-validation terbaik, sedangkan kurva final hanya menunjukkan proses optimasi saat model dilatih ulang untuk evaluasi akhir.
"""
path = NARRATIVE_DIR / "4_4_3_training_stability_interpretation.md"
path.write_text(interpretation_text, encoding="utf-8")
print(f"saved narrative: {path.relative_to(ROOT)}")
print(interpretation_text)
path


## Checklist 4.4.3

Memverifikasi artefak 4.4.3 yang diminta TODO.


In [ ]:
expected = [
    ("wajib", "Grafik loss training vs validation", FIG_DIR / "4_4_3_training_curves.png"),
    ("wajib", "Grafik IoU training vs validation", FIG_DIR / "4_4_3_training_curves.png"),
    ("wajib", "Grafik Dice training vs validation", FIG_DIR / "4_4_3_training_curves.png"),
    ("disarankan", "Kurva learning rate", FIG_DIR / "4_4_3_training_curves.png"),
    ("wajib", "Tabel titik kurva training", TABLE_DIR / "4_4_3_training_curve_points.csv"),
    ("wajib", "Tabel kebijakan training", TABLE_DIR / "4_4_3_training_policy_summary.csv"),
    ("disarankan", "Tabel ringkasan stabilitas per fold", TABLE_DIR / "4_4_3_training_stability_summary.csv"),
    ("disarankan", "Tabel ringkasan stabilitas per model", TABLE_DIR / "4_4_3_model_stability_summary.csv"),
    ("narasi_wajib", "Penjelasan checkpoint, loss, scheduler, early stopping, dan interpretasi kurva", NARRATIVE_DIR / "4_4_3_training_stability_interpretation.md"),
    ("opsional", "Kurva final training tanpa validation", FIG_DIR / "4_4_3_final_training_curves.png"),
]
checklist_4_4_3 = pd.DataFrame([
    {"priority": p, "todo_item": item, "artifact": str(path.relative_to(ROOT)), "exists": path.exists(), "size_bytes": path.stat().st_size if path.exists() else 0}
    for p, item, path in expected
])
save_table(checklist_4_4_3, "4_4_3_todo_coverage_checklist.csv")
missing = checklist_4_4_3[~checklist_4_4_3["exists"]]
if missing.empty:
    print("All 4.4.3 TODO artifacts created.")
else:
    print("Missing 4.4.3 artifacts:")
    display(missing)
